# Yorùbá OCR — Kaggle Notebook pipeline

**Runtime:** Kaggle Notebook → Accelerator → **GPU T4 x2** preferred, Internet **ON** for first setup/model downloads.

| Kaggle location | Path |
|-----------------|------|
| Working repo + outputs | `/kaggle/working/yoruba_ocr_research/` |
| Attached read-only datasets | `/kaggle/input/<dataset-slug>/` |
| Results | `/kaggle/working/yoruba_ocr_research/results/tables/` |
| HF cache | `/kaggle/working/yoruba_ocr_research/.hf_cache/` |

### Data upload pattern for Kaggle

1. Zip/upload your prepared dataset as a Kaggle Dataset, or create it from the Kaggle UI.
2. Attach that dataset to this notebook via **Add Input**.
3. The dataset should contain either:
   - `data/processed/...`, or
   - `processed/...`, or
   - files directly under a folder containing `labels/test.txt`.
4. Step §0.3 copies read-only Kaggle input data into `/kaggle/working/yoruba_ocr_research/data/processed/`.

### What to run

Edit **`RUN_PLAN`** in **§0.2** only. Everything else is Run All after setup.

| Section | What | Training? |
|---------|------|------------|
| **Setup** | Steps 0–5 — Kaggle paths, git pull, deps, data copy/check | — |
| **A** | Out-of-the-box baselines | No — PP-OCR EN, **PaddleOCR-VL-1.6 ZS**, **GLM-OCR ZS** |
| **B** | **PaddleOCR-VL-1.6 SFT** | Yes — optional supervised fine-tune |
| **D** | Analysis + compile Table 1 | No |
| *Appendix* | HF dataset + model uploads | Optional toggles in §0.2 |

**Kaggle secret:** add `HF_TOKEN` in Kaggle Notebook **Add-ons → Secrets** for gated/private model downloads or uploads.


## §0 Setup

### §0.1 Detect Kaggle runtime

Run this cell first. Kaggle has no Google Drive mount; `/kaggle/input` is read-only and `/kaggle/working` is writable.


In [ ]:
import os
import sys
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").is_dir()
IN_COLAB = False
KAGGLE_WORKING = Path("/kaggle/working") if IN_KAGGLE else Path.cwd()
KAGGLE_INPUT = Path("/kaggle/input") if IN_KAGGLE else Path.cwd() / "kaggle_input"
DRIVE_ROOT = None

if IN_KAGGLE:
    os.chdir(KAGGLE_WORKING)
    print("Kaggle runtime detected.")
    print("working:", KAGGLE_WORKING)
    print("input:", KAGGLE_INPUT)
else:
    print("Not in Kaggle — using local working directory for dry-run checks.")
    print("cwd:", Path.cwd())


### §0.2 Run plan (edit this only)

Kaggle note: keep Section B off unless you intentionally want a long GPU fine-tune run. Section A full VL/GLM eval also downloads large models.


In [ ]:
import os
import sys
from pathlib import Path

if "IN_KAGGLE" not in globals():
    IN_KAGGLE = Path("/kaggle/working").is_dir()
if "KAGGLE_WORKING" not in globals():
    KAGGLE_WORKING = Path("/kaggle/working") if IN_KAGGLE else Path.cwd()

# ── Edit these flags ─────────────────────────────────────────────────────────
RUN_PLAN = {
    # Section A: OOTB zero-shot baselines (PP-OCR EN, PaddleOCR-VL-1.6, GLM-OCR)
    "A_baselines_ootb": True,
    # Section B: PaddleOCR-VL-1.6 LM fine-tuning. Long GPU run; off if you only need baselines.
    "B_vl16_finetune": False,
    # Section C is legacy/classical PP-OCR ablations and remains off for the active paper plan.
    "C_ppocr_ablations": False,
    # Section D: Bootstrap CIs, stratified DER, compile CSVs
    "D_analysis_compile": True,
    # Appendix: HF dataset release
    "appendix_hf_dataset": False,
    # Appendix: HF model release
    "appendix_hf_models": False,
}

# Kaggle default: use attached processed data and avoid accidental resplitting.
os.environ.setdefault("USE_EXISTING_PROCESSED_DATA", "1")
os.environ.setdefault("RUN_RESPLIT", "0")
os.environ.setdefault("PYTHON", sys.executable)

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

try:
    from colab_run_plan import apply_run_plan, print_run_summary  # reused helper name
    env = apply_run_plan(RUN_PLAN)
    print_run_summary(RUN_PLAN)
except ImportError:
    print("RUN_PLAN configured. Helpers will load after §0.3 clones/updates the repo.")


### §0.3 Pull code from GitHub and copy Kaggle input data

This creates/updates `/kaggle/working/yoruba_ocr_research/` and copies attached read-only Kaggle data into the writable repo.

If your Kaggle Dataset slug/path is unusual, set `KAGGLE_PROCESSED_DATA_DIR` to the folder containing `labels/test.txt` before running this cell.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

os.environ["PYTHON"] = sys.executable

REPO_DIR_NAME = "yoruba_ocr_research"
GITHUB_REPO = "https://github.com/sam4rano/yoruba_ocr_research.git"
GITHUB_BRANCH = "main"

if "IN_KAGGLE" not in globals():
    IN_KAGGLE = Path("/kaggle/working").is_dir()
if "KAGGLE_WORKING" not in globals():
    KAGGLE_WORKING = Path("/kaggle/working") if IN_KAGGLE else Path.cwd()
if "KAGGLE_INPUT" not in globals():
    KAGGLE_INPUT = Path("/kaggle/input") if IN_KAGGLE else Path.cwd() / "kaggle_input"

repo = KAGGLE_WORKING / REPO_DIR_NAME if IN_KAGGLE else Path.cwd()
REPO_DIR = str(repo)
os.environ["PROJECT_ROOT"] = REPO_DIR
os.environ["HF_HOME"] = f"{REPO_DIR}/.hf_cache"
os.environ["HF_HUB_CACHE"] = os.environ["HF_HOME"]
os.environ.setdefault("PIP_CACHE_DIR", f"{REPO_DIR}/.pip_cache")


def _run(cmd: list[str], cwd: Path, *, check: bool = True) -> subprocess.CompletedProcess:
    res = subprocess.run(cmd, cwd=str(cwd), capture_output=True, text=True)
    if check and res.returncode != 0:
        raise RuntimeError(
            f"Command failed ({res.returncode}): {' '.join(cmd)}\n"
            f"STDOUT:\n{res.stdout}\nSTDERR:\n{res.stderr}"
        )
    return res


def _find_processed_data() -> Path | None:
    override = os.environ.get("KAGGLE_PROCESSED_DATA_DIR", "").strip()
    if override:
        p = Path(override)
        if (p / "labels" / "test.txt").is_file():
            return p
        raise FileNotFoundError(f"KAGGLE_PROCESSED_DATA_DIR does not contain labels/test.txt: {p}")

    candidates: list[Path] = []
    if KAGGLE_INPUT.is_dir():
        for root in sorted(KAGGLE_INPUT.iterdir()):
            candidates.extend([
                root / "data" / "processed",
                root / "processed",
                root,
            ])
            candidates.extend(root.glob("**/data/processed"))
            candidates.extend(p.parent.parent for p in root.glob("**/labels/test.txt"))
    for p in candidates:
        if (p / "labels" / "test.txt").is_file():
            return p
    return None


def _copy_processed_data(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and (dst / "labels" / "test.txt").is_file():
        print(f"Using existing writable data/processed at {dst}")
        return
    print(f"Copying Kaggle input data: {src} -> {dst}")
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)


repo.parent.mkdir(parents=True, exist_ok=True)
if not (repo / ".git").is_dir():
    if repo.exists() and any(repo.iterdir()):
        print(f"Repo folder exists without .git; preserving data/results and initializing git at {repo}")
        _run(["git", "init"], repo)
        remotes = _run(["git", "remote"], repo, check=False).stdout.split()
        if "origin" not in remotes:
            _run(["git", "remote", "add", "origin", GITHUB_REPO], repo)
    else:
        print(f"Cloning repository to {repo}...")
        _run(["git", "clone", "--depth", "1", "-b", GITHUB_BRANCH, GITHUB_REPO, str(repo)], KAGGLE_WORKING)
else:
    print(f"Existing git repository detected at {repo}. Updating remote URL...")
    _run(["git", "remote", "set-url", "origin", GITHUB_REPO], repo)

print(f"Fetching origin/{GITHUB_BRANCH}...")
fetch = _run(["git", "fetch", "--depth", "1", "origin", GITHUB_BRANCH], repo, check=False)
if fetch.returncode != 0:
    raise RuntimeError(
        f"git fetch failed. Ensure Kaggle Internet is ON.\nSTDOUT:\n{fetch.stdout}\nSTDERR:\n{fetch.stderr}"
    )
_run(["git", "checkout", "-f", "-B", GITHUB_BRANCH, f"origin/{GITHUB_BRANCH}"], repo)
_run(["git", "reset", "--hard", f"origin/{GITHUB_BRANCH}"], repo)

os.chdir(repo)
DATA_DIR = repo / "data"
processed_src = _find_processed_data()
if processed_src is not None:
    _copy_processed_data(processed_src, DATA_DIR / "processed")
else:
    print("No attached Kaggle data/processed found. Upload/attach a Kaggle Dataset or run consolidation from data/raw.")

head = _run(["git", "rev-parse", "--short", "HEAD"], repo).stdout.strip()
print("cwd:", os.getcwd())
print("git HEAD:", head)
print("data/processed:", (DATA_DIR / "processed").is_dir())
print("data/raw:", (DATA_DIR / "raw").is_dir())

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from colab_stream import run_cmd, run_phase
from colab_run_plan import apply_run_plan, print_run_summary

apply_run_plan(RUN_PLAN)
print_run_summary(RUN_PLAN)
print("run_cmd / run_phase ready — long steps stream logs live.")

test_labels = DATA_DIR / "processed" / "labels" / "test.txt"
if os.environ.get("USE_EXISTING_PROCESSED_DATA", "1") == "1" and test_labels.is_file():
    os.environ["SKIP_CONSOLIDATE"] = "1"
    os.environ["RUN_RESPLIT"] = "0"
    with test_labels.open(encoding="utf-8") as fh:
        n = sum(1 for _ in fh)
    print(f"Using existing data/processed (test.txt: {n} lines). SKIP_CONSOLIDATE=1")


### §0.4 Install dependencies

Kaggle notes:
- Turn **Internet ON** before this cell.
- Kaggle images already include CUDA/PyTorch, but PaddleOCR and Transformers still need project-specific installs.
- If Paddle GPU wheel selection fails, use the official Paddle install selector for the current Kaggle CUDA version.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path("requirements.txt").is_file():
    raise RuntimeError("Run §0.3 (git pull + data copy) first.")

PY = sys.executable


def pip(*args):
    subprocess.check_call([PY, "-m", "pip", "install", "-q", *args])

# Paddle GPU wheel. Kaggle currently uses Linux+CUDA; this install may change as Kaggle images change.
# If it fails, enable Internet and install the wheel recommended by Paddle's official selector.
if IN_KAGGLE:
    try:
        pip("paddlepaddle-gpu", "-f", "https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html")
    except Exception as exc:
        print("WARNING: paddlepaddle-gpu install failed:", exc)
        print("Try the current official Paddle GPU wheel command for Kaggle's CUDA image.")

processed_reqs = []
for ln in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    ln = ln.strip()
    if not ln or ln.startswith("#"):
        continue
    if ln.startswith("paddlepaddle"):
        continue
    if "editdistance" in ln:
        ln = "editdistance>=0.6.2"
    if "opencv-python" in ln:
        continue
    if "pandas" in ln:
        ln = "pandas>=2.0.0"
    if "accelerate" in ln:
        ln = "accelerate>=1.1.0"
    if "transformers" in ln:
        ln = "transformers>=5"
    processed_reqs.append(ln)

Path("/tmp/reqs_no_paddle.txt").write_text("\n".join(processed_reqs) + "\n", encoding="utf-8")
pip("-r", "/tmp/reqs_no_paddle.txt")
pip("transformers>=5", "accelerate>=1.1.0", "datasets", "safetensors", "huggingface_hub>=1.5.0")
pip("bitsandbytes")
pip("numpy>=1.24.0,<2.0.0", "pandas>=2.0.0", "--force-reinstall")

# Kaggle secrets: Add-ons → Secrets → HF_TOKEN
try:
    from kaggle_secrets import UserSecretsClient  # type: ignore
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
        os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", tok)
        from huggingface_hub import login
        login(token=tok)
        print("HF_TOKEN loaded from Kaggle secret")
except Exception as exc:
    print("HF_TOKEN not set via Kaggle secrets:", exc)

try:
    import paddle
    print("Paddle", paddle.__version__, "CUDA", paddle.device.is_compiled_with_cuda())
except Exception as exc:
    print("Paddle import failed:", exc)

import torch
print("Torch", torch.__version__, "CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


### §0.5 Clone PaddleOCR + extras


In [ ]:
import shutil, subprocess, sys
from pathlib import Path

pdir = Path("PaddleOCR")
if pdir.is_dir() and not (pdir / "requirements.txt").is_file():
    shutil.rmtree(pdir)
if not pdir.is_dir():
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "main",
                           "https://github.com/PaddlePaddle/PaddleOCR.git", "PaddleOCR"])
if (pdir / "requirements.txt").is_file():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "PaddleOCR/requirements.txt"])

# PaddleOCR requirements can pull older transitive deps. Re-assert the VLM stack.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=5.0.0", "accelerate>=1.1.0", "huggingface_hub>=1.5.0",
    "einops", "torchvision", "bitsandbytes"
])
print("PaddleOCR OK; VLM dependency stack re-checked")


### §0.6 Pre-flight smoke test (no GPU training)

Run **after Step 3** (deps installed). Uses `--quick`: data layout, config, VL JSONL export — **does not** require eval JSONL yet. For the old heavy check (analysis 17–19, HF card), pass `--full` after Step 11.


In [ ]:
import os, subprocess, sys
from pathlib import Path

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from colab_stream import run_cmd

repo = Path(REPO_DIR)
os.environ["PYTHON"] = sys.executable
os.environ["PROJECT_ROOT"] = str(repo)
run_cmd(
    [
        sys.executable,
        "scripts/24_colab_smoke_test.py",
        "--quick",
        "--skip-config",
        "--network",
    ],
    cwd=repo,
)
print("Smoke + network checks passed — safe to continue to baselines .")


## §1 Data & Config (Phases 01–03)

With `USE_EXISTING_PROCESSED_DATA=1` (default), **Phase 01 is skipped** and your Drive `data/processed/` is used as-is.

Set `SKIP_CONSOLIDATE=0` and `RUN_RESPLIT=1` in Step 0 only when rebuilding from `data/raw/`.


In [ ]:
import os, subprocess, sys, json, shutil
from pathlib import Path

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from colab_stream import run_cmd, run_phase

PY = sys.executable
Path("results/tables").mkdir(parents=True, exist_ok=True)

if os.environ.get("SKIP_CONSOLIDATE", "1") == "1":
    print("SKIP_CONSOLIDATE=1 — using existing data/processed on Drive (not creating empty data/raw)")
else:
    Path("data/raw").mkdir(parents=True, exist_ok=True)
    if os.environ.get("RESET_PROCESSED", "0") == "1" and Path("data/processed").is_dir():
        shutil.rmtree("data/processed")
    cmd = [PY, "scripts/01_consolidate_data.py",
           "--raw-dir", "data/raw", "--output-dir", "data/processed",
           "--log-file", "results/tables/consolidation_report.json"]
    if os.environ.get("RUN_RESPLIT", "0") == "1":
        cmd += ["--resplit", "--seed", "42", "--train-ratio", "0.8", "--val-ratio", "0.1", "--test-ratio", "0.1"]
    run_cmd(cmd)

checks = {
    "data/processed/labels/train.txt": "file",
    "data/processed/labels/val.txt": "file",
    "data/processed/labels/test.txt": "file",
    "data/processed/dictionary/yoruba_char_dict.txt": "file",
    "data/processed/images/test": "dir",
}
for path, kind in checks.items():
    p = Path(path)
    ok = p.is_file() if kind == "file" else p.is_dir()
    extra = ""
    if ok and kind == "file":
        with p.open(encoding="utf-8") as fh:
            line_count = sum(1 for _ in fh)
        extra = f" ({line_count} lines)"
    elif ok:
        valid_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}
        img_files = [f for f in p.glob("*") if f.suffix.lower() in valid_exts]
        extra = f" ({len(img_files)} images)"
    print("OK" if ok else "MISSING", path, extra)
    if not ok:
        raise RuntimeError("Upload data to Drive under data/raw/ or data/processed/")

rep = Path("results/tables/consolidation_report.json")
if rep.is_file():
    try:
        r = json.loads(rep.read_text(encoding="utf-8"))
        sc = r.get("split_counts", {})
        print(f"consolidation: n={r.get('unique_images_total')} train={sc.get('train')} val={sc.get('val')} test={sc.get('test')}")
    except json.JSONDecodeError as exc:
        print(f"WARNING: consolidation_report.json is malformed or partially written: {exc}")

run_cmd([PY, "scripts/02c_refresh_dataset_report.py"])
run_cmd([PY, "scripts/02b_data_quality_audit.py", "--data-dir", "data/processed", "--out-json", "results/tables/data_quality.json"])
run_phase("phase_02_analyze.sh")
env = os.environ.copy()
env["CONFIG_FORCE_GPU"] = "1"
run_phase("phase_03_config.sh", env=env)
print("Phases 01-03 done")


In [ ]:
# Re-run data quality audit after Phase 02 to ensure image dimensions are captured
import sys
from colab_stream import run_cmd  # noqa: F811
run_cmd([sys.executable, "scripts/02b_data_quality_audit.py", "--data-dir", "data/processed", "--out-json", "results/tables/data_quality.json"])


In [ ]:
# Freeze split labels to data/splits/ for reproducibility if not already done
import shutil
from pathlib import Path
splits_dir = Path("data/splits")
splits_dir.mkdir(parents=True, exist_ok=True)
for split in ("train", "val", "test"):
    src = Path("data/processed/labels") / f"{split}.txt"
    dest = splits_dir / f"{split}.txt"
    if src.is_file():
        shutil.copy2(src, dest)
        print(f"Frozen {split}.txt → {dest}")


## §2 Baselines (Section A — OOTB zero-shot evals)

PP-OCR English pretrained · **PaddleOCR-VL-1.6 zero-shot** · **GLM-OCR zero-shot**

Skipped when `A_baselines_ootb=False`.


In [ ]:
import gc, os, subprocess, sys
from pathlib import Path

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from colab_stream import run_cmd, run_phase

try:
    from IPython.display import display
except ImportError:
    display = print

if os.environ.get("RUN_PLAN_A", "1") != "1":
    print("Section A skipped (A_baselines_ootb=False)")
else:
    env = os.environ.copy()
    if os.environ.get("ARCHIVE_METRICS_BEFORE_RUN", "0") == "1":
        run_cmd([sys.executable, "scripts/metrics_lifecycle.py", "archive"])

    env.setdefault("SKIP_PADDLE_FINETUNE", "1")
    env["EVAL_USE_GPU"] = "1"

    # 1. Base PaddleOCR -- PP-OCR English pretrained
    run_phase("phase_05_eval_paddleocr_recognition.sh", env=env, label="PaddleOCR EN pretrained")

    # Flush Paddle CUDA context before loading PyTorch VLM models
    # to prevent torch.cuda.is_available() from returning False.
    gc.collect()
    try:
        import torch as _t
        if _t.cuda.is_available():
            _t.cuda.empty_cache()
    except Exception:
        pass

    # 2. PaddleOCR-VL-1.6 zero-shot
    # PADDLEOCRVL16_QUANTIZE_4BIT=0 (default): use hardware-native dtype (bf16 on L4/A100, fp16 on T4).
    # PADDLEOCRVL16_QUANTIZE_4BIT=1:           use 4-bit only when VRAM < 6 GB.
    if env.get("SKIP_PADDLEOCRVL16_ZERO_SHOT", "0") == "0":
        env["PADDLEOCRVL16_QUANTIZE_4BIT"] = env.get("PADDLEOCRVL16_QUANTIZE_4BIT", "0")
        # Ensure no stale sample-cap from previous debug runs leaks into full eval.
        env.pop("PADDLEOCRVL16_MAX_SAMPLES", None)
        run_phase("phase_15_eval_paddleocrvl16_zero_shot.sh", env=env, label="PaddleOCR-VL-1.6 zero-shot")
    else:
        print("SKIP_PADDLEOCRVL16_ZERO_SHOT=1")

    # Flush CUDA cache between large VLM loads
    gc.collect()
    try:
        if _t.cuda.is_available():
            _t.cuda.empty_cache()
    except Exception:
        pass

    # 3. GLM-OCR zero-shot
    # GLM_QUANTIZE_4BIT=0 (default): use hardware-native dtype (bf16 on L4/A100, fp16 on T4).
    if env.get("SKIP_GLM_ZERO_SHOT", "0") == "0":
        env["GLM_QUANTIZE_4BIT"] = env.get("GLM_QUANTIZE_4BIT", "0")
        # Ensure no stale sample-cap from previous debug runs leaks into full eval.
        env.pop("GLM_MAX_SAMPLES", None)
        run_phase("phase_18_eval_glm_ocr_zero_shot.sh", env=env, label="GLM-OCR zero-shot")
    else:
        print("SKIP_GLM_ZERO_SHOT=1")

    import pandas as pd
    from pathlib import Path
    mp = Path("results/tables/metrics.csv")
    if mp.is_file():
        df = pd.read_csv(mp)
        display(df.tail(10))


## §3 Fine-tuned Models (Section B — VL-1.6 SFT)

### §3.1 PaddleOCR-VL-1.6 SFT

**B1 — PaddleOCR-VL-1.6:** export JSONL → zero-shot eval → **SFT train** → SFT eval.

Skipped when `B_finetuned=False`.


In [ ]:
import os, sys
from pathlib import Path

import pandas as pd

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from colab_stream import run_phase

if os.environ.get("RUN_PLAN_B", "1") != "1":
    print("Phase 14 export skipped (B_vl16_finetune=False)")
else:
    run_phase("phase_14_export_paddleocrvl16_sft.sh", label="VL-1.6 SFT export")
    for split in ("train", "val", "test"):
        f = Path("data/paddleocrvl16_sft") / f"{split}.jsonl"
        if f.is_file():
            with f.open(encoding="utf-8") as fh:
                n = sum(1 for _ in fh)
        else:
            n = 0
        print(f"{split}.jsonl: {n}")

if os.environ.get("SKIP_PADDLEOCRVL16_ZERO_SHOT", "1") == "0":
    os.environ["SKIP_PADDLEOCRVL16_ZERO_SHOT"] = "0"
    os.environ["PADDLEOCRVL16_QUANTIZE_4BIT"] = os.environ.get("PADDLEOCRVL16_QUANTIZE_4BIT", "0")
    os.environ.pop("PADDLEOCRVL16_MAX_SAMPLES", None)  # ensure full n=326 eval
    run_phase("phase_15_eval_paddleocrvl16_zero_shot.sh", label="VL-1.6 zero-shot eval")
else:
    print("VL-1.6 zero-shot skipped (Section B off or SKIP_PADDLEOCRVL16_ZERO_SHOT=1)")

mp = Path("results/tables/metrics.csv")
if mp.is_file():
    df = pd.read_csv(mp)
    col = "model_name" if "model_name" in df.columns else "model"
    zs = df[df[col] == "paddleocrvl16_zero_shot"]
    print("zero-shot row:", "OK" if not zs.empty else "MISSING")


In [ ]:
import os, sys
from pathlib import Path

import pandas as pd

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from colab_stream import run_cmd, run_phase

try:
    from IPython.display import display
except ImportError:
    display = print

if os.environ.get("RUN_PLAN_B", "1") != "1":
    print("Section B PaddleOCR-VL-1.6 SFT skipped (B_vl16_finetune=False)")
else:
    # VRAM pre-flight check — SFT training is the heaviest GPU phase
    try:
        import torch as _torch
        if _torch.cuda.is_available():
            _free, _total = _torch.cuda.mem_get_info()
            _free_gb  = _free  / (1024 ** 3)
            _total_gb = _total / (1024 ** 3)
            print(f"VRAM: {_free_gb:.1f} GB free / {_total_gb:.1f} GB total")
            if _free_gb < 8.0:
                print(
                    f"WARNING: Only {_free_gb:.1f} GB VRAM free. "
                    "VL-1.6 4-bit SFT typically needs ≥8 GB. "
                    "Consider restarting the runtime to free memory before training."
                )
        else:
            print("WARNING: No CUDA GPU detected — training will run on CPU (very slow).")
    except ImportError:
        print("torch not yet installed — VRAM check skipped.")
    env = {**os.environ, "EVAL_USE_GPU": "1", "PYTHONUNBUFFERED": "1"}
    ckpt_dir = Path("experiments/paddleocrvl16_sft")
    if ckpt_dir.is_dir() and (ckpt_dir / "training_state.json").is_file():
        print(f"Found existing fine-tuning checkpoint state at {ckpt_dir}.")
        print("Resuming training from the last saved state (PADDLEOCRVL16_SFT_RESUME=1).")
        env["PADDLEOCRVL16_SFT_RESUME"] = "1"

    if os.environ.get("SKIP_PADDLEOCRVL16_SFT_TRAIN", "0") == "0":
        # Remove sample-cap from the env snapshot passed to training.
        # Verified: phase_16_train_paddleocrvl16_sft.sh reads PADDLEOCRVL16_SFT_MAX_SAMPLES;
        # phase_17 uses PADDLEOCRVL16_EVAL_MAX_SAMPLES — no cross-phase leakage.
        env.pop("PADDLEOCRVL16_SFT_MAX_SAMPLES", None)
        run_phase("phase_16_train_paddleocrvl16_sft.sh", env=env, label="Phase 16 VL-1.6 Fine-Tuning")
    else:
        print("SKIP_PADDLEOCRVL16_SFT_TRAIN=1")

    # Evaluate the fine-tuned model
    if os.environ.get("SKIP_PADDLEOCRVL16_SFT_TRAIN", "0") == "0":
        run_phase("phase_17_eval_paddleocrvl16_sft.sh", env=env, label="VL-1.6 fine-tuned eval")

    mp = Path("results/tables/metrics.csv")
    if mp.is_file():
        df = pd.read_csv(mp)
        col = "model_name" if "model_name" in df.columns else "model"
        vl = df[df[col].str.startswith("paddleocrvl16", na=False)]
        display(vl[[c for c in (col, "n", "cer", "wer", "der") if c in vl.columns]].to_string(index=False))


## §4 Optional Classical PaddleOCR Fine-Tune

This is an optional supervised PaddleOCR recognition comparison, not a default paper section. It uses `scripts/train_paddleocr_recognition.py`, which delegates to `PaddleOCR/tools/train.py` and writes `results/tables/train_run.json`.

Default: skipped with `SKIP_PADDLE_TRAIN=1`. Set `SKIP_PADDLE_TRAIN=0` only when you intentionally want the long classical training run.


In [ ]:
import os, sys
from pathlib import Path

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from colab_stream import run_phase

if os.environ.get("SKIP_PADDLE_TRAIN", "1") != "0":
    print("SKIP_PADDLE_TRAIN=1 — skipping optional PaddleOCR recognition fine-tune")
else:
    print("Phase 04: optional PaddleOCR recognition fine-tune. Logs stream below.")
    sys.stdout.flush()
    env = {
        **os.environ,
        "PYTHON": sys.executable,
        "PYTHONUNBUFFERED": "1",
        "EVAL_USE_GPU": "1",
        "CONFIG_FORCE_GPU": "1",
        "TRAIN_RESUME": "1",
    }
    run_phase("phase_04_train_paddleocr_recognition.sh", env=env, label="Phase 04 PaddleOCR recognition fine-tune")
    print("Phase 04 finished. See results/tables/train_run.json for timing.")


### §4.2 Ablations

Ablation tables from earlier pilot runs have been removed from the active notebook because their runner scripts are not part of the current reproducible pipeline. Add a new ablation script and fresh metrics before restoring this section.


In [ ]:
print("No active ablation phase. Current notebook proceeds to Section D analysis and compile.")


## §5 Analysis & Tables (Section D — stratified, bootstrap, compile)

### §5.1 Bootstrap & Stratified Analysis

Bootstrap CIs, stratified DER, DER-universe ablation, then Table 1. These scripts require fresh JSONL logs from the active model rows.

Skipped when `D_analysis_compile=False`.


In [ ]:
import os, sys
from pathlib import Path

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from colab_stream import run_cmd

if os.environ.get("RUN_PLAN_D", "1") != "1":
    print("Section D skipped (D_analysis_compile=False)")
elif os.environ.get("SKIP_ANALYSIS", "0") == "1":
    print("SKIP_ANALYSIS=1")
else:
    for s in ("17_stratified_error_analysis.py", "18_der_universe_ablation.py", "19_bootstrap_metric_cis.py"):
        run_cmd([sys.executable, f"scripts/{s}"], label=s)


### §5.2 Compile & Audit

Runs only when Section D is enabled.


In [ ]:
import json, os, sys
from pathlib import Path

import pandas as pd

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from colab_stream import run_cmd, run_phase

try:
    from IPython.display import display
except ImportError:
    display = print

if os.environ.get("RUN_PLAN_D", "1") != "1":
    print("Section D compile skipped (D_analysis_compile=False)")
else:
    run_phase("phase_09_compile.sh", label="Compile paper tables")

    run_cmd([
        sys.executable, "scripts/12_diagnose_hypotheses.py", "checkpoints",
        "--csv", "results/tables/metrics.csv",
        "--report", "results/tables/checkpoint_audit.json",
    ])

    expected = [
        "table1_main_comparison.csv",
        "metrics_summary.csv",
        "bootstrap_metric_cis.csv",
        "bootstrap_pairwise_comparison.csv",
        "stratified_der_by_density.csv",
        "stratified_by_linguistic_features.csv",
        "der_universe_ablation.csv",
        "error_taxonomy.csv",
    ]
    for name in expected:
        p = Path("results/tables") / name
        print(("OK" if p.is_file() else "MISSING"), name)

    # Verify generated plots
    expected_figs = [
        "model_metrics_comparison.png",
        "bootstrap_confidence_intervals.png",
        "stratified_der_by_density.png",
        "error_taxonomy_distribution.png",
        "hard_cases_benchmark.png",
    ]
    for fig in expected_figs:
        p = Path("results/tables/figures") / fig
        print(("OK" if p.is_file() else "MISSING"), f"figure: {fig}")

    try:
        df = pd.read_csv("results/tables/metrics_summary.csv")
    except Exception as e:
        raise RuntimeError(
            f"Could not read metrics_summary.csv — re-run Section D: {e}"
        ) from e
    display(df)
    # Display linguistic features stratification
    lf_csv = Path("results/tables/stratified_by_linguistic_features.csv")
    if lf_csv.is_file():
        print("\n=== Stratified by Linguistic Features ===")
        display(pd.read_csv(lf_csv))

    if "phantom" in df.columns and df["phantom"].astype(str).str.lower().isin(["true", "1"]).any():
        print("WARNING: phantom=true rows — do not cite")

    audit = Path("results/tables/checkpoint_audit.json")
    if audit.is_file():
        try:
            a = json.loads(audit.read_text(encoding="utf-8"))
            bad = [r for r in a.get("rows", []) if r.get("status") != "ok"]
            print("checkpoint audit:", "OK" if not bad else f"{len(bad)} issue(s)")
        except json.JSONDecodeError as exc:
            print(f"WARNING: checkpoint_audit.json is malformed or partially written: {exc}")

    rep = Path("results/tables/eval_alignment_report.json")
    if rep.is_file():
        try:
            r = json.loads(rep.read_text(encoding="utf-8"))
            print("eval alignment mismatches:", len(r.get("mismatches", [])))
        except json.JSONDecodeError as exc:
            print(f"WARNING: eval_alignment_report.json is malformed or partially written: {exc}")


### §5.3 Hard-Cases Benchmark

Evaluates model performance on **four linguistically challenging sub-categories** of the Yorùbá test set. These are defined purely from the ground-truth text — no manual annotation required — making them fully reproducible.

| Category | Detection Rule |
|---|---|
| **Named Entities** | Capitalised word mid-sentence OR any capitalised word carrying a diacritic |
| **Numerics** | Line contains at least one digit (currency, dates, percentages) |
| **Historical Orthography** | Contains `sh` or apostrophe-adjoined characters (pre-1974 conventions) |
| **Code-Mixed (Yorùbá–English)** | Line has ≥ 1 diacritic-bearing (Yorùbá) word **and** ≥ 1 pure-ASCII word (len ≥ 3) |

> **Figure 5** shows CER (%) per model across all four categories. Categories with higher CER reveal where each model has structural weaknesses beyond overall performance.

> **Overlap note:** Categories are **intentionally non-exclusive** — a single line may match multiple categories simultaneously (e.g. a line with a digit *and* a capitalised diacritic word matches both *Numerics* and *Named Entities*). Each category is evaluated independently over all matching lines; CER figures are not deduplicated across categories. This is by design — we report per-category difficulty, not a mutually-exclusive partition. See `17_stratified_error_analysis.py` → `write_features_csv()` for the aggregation logic.


In [ ]:
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display, Image as IPImage
except ImportError:
    display = print
    IPImage = None

# ── Hard-Cases Benchmark table ──────────────────────────────────────────
features_csv = Path("results/tables/stratified_by_linguistic_features.csv")
if features_csv.is_file():
    df_feat = pd.read_csv(features_csv)
    print("=== Hard-Cases Benchmark: CER/WER/DER by Linguistic Feature ===")
    # Pivot to wide form: feature × model for easier comparison
    try:
        pivot = df_feat.pivot_table(
            index="feature",
            columns="model",
            values="cer_pct",
            aggfunc="first",
        )
        display(pivot)
    except Exception as e:
        print(f"Pivot skipped ({e}); showing raw table instead.")
        display(df_feat)
else:
    print("stratified_by_linguistic_features.csv not yet generated — run Phase 17 first.")

# ── Figure 5: Hard-Cases Benchmark bar chart ─────────────────────────────
fig5 = Path("results/tables/figures/hard_cases_benchmark.png")
if fig5.is_file():
    print("\nFigure 5: Hard-Cases Benchmark — CER by Linguistic Feature Category")
    if IPImage is not None:
        display(IPImage(filename=str(fig5), width=900))
    else:
        print(f"[Figure saved at {fig5}]")
else:
    print("hard_cases_benchmark.png not yet generated — run Phase 22 (generate_plots) first.")


## §6 Publish

### §6.1 Write research_approach.md


In [ ]:
import sys
from pathlib import Path

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from colab_stream import run_cmd

out = Path("research_approach.md")
run_cmd([sys.executable, "scripts/23_write_research_approach.py", "--output", str(out)])
print("Saved:", out.resolve())
print(out.read_text(encoding="utf-8")[:2000], "...")


### §6.2 Save/backup results on Kaggle

Kaggle persists files under `/kaggle/working` as notebook outputs. Use **Save Version** to keep `results/`, `experiments/`, and generated tables.


In [ ]:
from pathlib import Path

root = Path.cwd()
print("Kaggle working repo:", root)
for rel in ["results/tables", "experiments", "data/hf_export"]:
    p = root / rel
    print(f"{rel}:", "exists" if p.exists() else "missing")

print()
print("To persist outputs: Kaggle Notebook → Save Version → include output files.")
print("To download manually: use the file browser on /kaggle/working/yoruba_ocr_research.")


### §6.3 Hugging Face releases

Set `appendix_hf_dataset=True` and/or `appendix_hf_models=True` in §0.2. Requires Kaggle secret **`HF_TOKEN`** with write access.

- **Dataset** — full benchmark (`data/processed/`) via `25_upload_hf_dataset.py`
- **Models** — VL-1.6 SFT adapter


In [ ]:
import os, sys
from pathlib import Path

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
from colab_stream import run_cmd

if os.environ.get("SKIP_HF_DATASET_UPLOAD", "1") == "1":
    print("SKIP_HF_DATASET_UPLOAD=1 — preview card with:")
    print("  python scripts/25_upload_hf_dataset.py --dry-run")
else:
    repo = Path(REPO_DIR)
    cmd = [sys.executable, "scripts/25_upload_hf_dataset.py", "--push"]
    repo_id = os.environ.get("HF_DATASET_REPO_ID", "").strip()
    if repo_id:
        cmd.extend(["--repo-id", repo_id])
    if os.environ.get("HF_DATASET_PRIVATE", "0") == "1":
        cmd.append("--private")
    run_cmd(cmd, cwd=repo, label="HF dataset upload")
    manifest = repo / "results/tables/hf_dataset_upload.json"
    if manifest.is_file():
        print(manifest.read_text(encoding="utf-8"))


### §6.4 Upload fine-tuned models

Pushes VL SFT when `appendix_hf_models=True`.


In [ ]:
import os, sys
from pathlib import Path

try:
    from colab_stream import run_cmd
except ImportError:
    from subprocess import run as run_cmd

if os.environ.get("SKIP_HF_MODELS_UPLOAD", "1") == "1":
    print("SKIP_HF_MODELS_UPLOAD=1 — model upload skipped.")
    print("To upload, set appendix_hf_models=True in §0.2 RUN_PLAN and re-run.")
    print("Manual alternative: huggingface-cli upload <repo_id> experiments/paddleocrvl16_sft/")
else:
    repo_id = os.environ.get("HF_MODEL_REPO_ID", "").strip()
    if not repo_id:
        raise ValueError(
            "Set HF_MODEL_REPO_ID env var (e.g. 'sam4rano/paddleocr-vl16-yoruba') before uploading."
        )
    model_dir = Path("experiments/paddleocrvl16_sft")
    if not model_dir.is_dir():
        raise FileNotFoundError(f"Fine-tuned model not found at {model_dir} — run Section B first.")
    cmd = [
        "huggingface-cli", "upload",
        repo_id,
        str(model_dir),
        "--repo-type", "model",
    ]
    if os.environ.get("HF_MODEL_PRIVATE", "0") == "1":
        cmd += ["--private"]
    run_cmd(cmd)
    print(f"Model uploaded to https://huggingface.co/{repo_id}")


### §7.1 Inference demo


In [ ]:
import gc, os
import sys
from pathlib import Path
import torch
from PIL import Image

try:
    from IPython.display import display
except ImportError:
    display = print

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_mem, total_mem = torch.cuda.mem_get_info()
    free_gb = free_mem / (1024 ** 3)
    total_gb = total_mem / (1024 ** 3)
    print(f"CUDA VRAM Info: Free: {free_gb:.2f} GB / Total: {total_gb:.2f} GB")
    if free_gb < 4.0:
        print("WARNING: Low GPU memory. If another model is active, delete it or restart the runtime.")

scripts_path = str(Path.cwd() / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)
import paddle_vl_shared

from transformers import AutoProcessor, AutoModel

base_id = "PaddlePaddle/PaddleOCR-VL-1.6"
model_local = Path("experiments/paddleocrvl16_sft")
model_path = str(model_local) if model_local.is_dir() else base_id

test_images = sorted(
    img
    for ext in ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff")
    for img in Path("data/processed/images/test").glob(ext)
)
if not test_images:
    print("No test images found — run Section 1 first.")
else:
    test_image = test_images[0]
    print(f"Sample image: {test_image.name}")
    print(f"Loading model from: {model_path}")

    # Note: trust_remote_code=True is required because custom vision-language models
    # define their specific architectures/layers in Python code hosted on HF Hub.
    processor = AutoProcessor.from_pretrained(base_id, trust_remote_code=True)
    model = AutoModel.from_pretrained(
        model_path,
        trust_remote_code=True,
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    ).eval()

    image = Image.open(test_image).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": paddle_vl_shared.USER_TEXT_OCR_YORUBA},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    device = next(model.parameters()).device
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=128)
        trimmed = [ids[inputs.input_ids.shape[1]:] for ids in out_ids]
        prediction = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

    print(f"Model   : {base_id} (zero-shot/fine-tuned)")
    print(f"Prediction: {prediction}")
    display(image)


In [ ]:
# Final Sanity Check: Table 1 Validation
import pandas as pd
from pathlib import Path

try:
    from IPython.display import display
except ImportError:
    display = print

summary_path = Path("results/tables/metrics_summary.csv")
if not summary_path.is_file():
    print("❌ metrics_summary.csv not found! Run Section D (compile) first.")
else:
    try:
        df = pd.read_csv(summary_path)
    except Exception as e:
        raise RuntimeError(
            f"Could not read metrics_summary.csv — re-run Section D: {e}"
        ) from e
    print("=== Table 1 — Compiled Metrics Summary ===")
    print(df.to_string(index=False))
    print("\n=== Expected Model Row Checklist ===")

    # Active model rows in the paper
    expected_models = [
        ("paddleocr_en_pretrained", "PaddleOCR EN pretrained"),
        ("paddleocrvl16_zero_shot",   "PaddleOCR-VL-1.6 (zero-shot)"),
        ("glm_ocr_zero_shot",          "GLM-OCR (zero-shot)"),
        ("paddleocrvl16_sft", "PaddleOCR-VL-1.6 (SFT, optional)"),
    ]

    missing = []
    for model_key, display_name in expected_models:
        present = model_key in df["model_label"].values
        status = "✅ PASS" if present else "⏳ PENDING"
        if not present:
            missing.append(model_key)
        print(f"{status:<10} | {display_name}")

    print()
    if not missing:
        print("✅ All active model rows present — Table 1 is complete.")
    else:
        print("⏳ Clean active results are missing model rows that still need full evaluation:")
        for model_key in missing:
            print(f"  - {model_key}")
        print("Re-run Section A evaluation(s) on GPU, then Section D compile/alignment.")
